# 01e — Essentia MuSE Models (MusiCNN & VGGish) + Valence Key Correction

Benchmarks the two Essentia **MuSE** regression heads — `muse-msd-musicnn-2` and
`muse-audioset-vggish-2` — against the three available datasets (DEAM, PMEmo, MERGE),
in both base and key-corrected configurations.

The MuSE dataset is a third large-scale music emotion corpus distinct from EmoMusic
and DEAM. All MuSE heads share the same architecture and [1, 9] output range as the
existing emomusic and deam heads, making them direct drop-in replacements.

**4 model variants evaluated:**

| Model | Backbone | Head | Key correction |
|---|---|---|---|
| muse-musicnn | MusiCNN | muse-msd-musicnn-2 | No |
| muse-musicnn+key | MusiCNN | muse-msd-musicnn-2 | Yes (α=0.13) |
| muse-vggish | VGGish | muse-audioset-vggish-2 | No |
| muse-vggish+key | VGGish | muse-audioset-vggish-2 | Yes (α=0.13) |

Key correction: `valence += α × (is_major − 0.5) × key_strength`  
α = 0.13 (same value validated on the production EmoMusic model in 01c/01d).

Two-phase evaluation:
1. **Backbone extraction** — embeddings + key info extracted once per audio file and cached in memory.
2. **Head evaluation** — each of the 4 variants scored against cached embeddings (fast).

## Section 0 — Setup

In [ ]:
!pip install essentia-tensorflow yt-dlp librosa matplotlib pandas scipy tqdm gdown

## Section 1 — Shared utilities

In [ ]:
import os, sys, pickle, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# ── Paths (Colab / Kaggle / local) ───────────────────────────────────────────
def _find_eval_dir():
    for candidate in [
        Path('/kaggle/working/Soundtrack-Mood-Manager/evaluation'),
        Path('/content/Soundtrack-Mood-Manager/evaluation'),
        Path(__file__).resolve().parent if '__file__' in dir() else Path('.'),
        Path('.'),
    ]:
        if (candidate / 'eval_datasets.py').exists():
            return candidate
    raise FileNotFoundError('Cannot locate evaluation/ directory')

EVAL_DIR   = _find_eval_dir()
REPO_ROOT  = EVAL_DIR.parent
DATA_DIR   = EVAL_DIR / 'data'
MODELS_DIR = REPO_ROOT / 'models'
SPOTCHECK_DIR = EVAL_DIR / 'spotchecks'

for d in (DATA_DIR, MODELS_DIR, SPOTCHECK_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(EVAL_DIR))
from eval_datasets  import setup_deam, setup_pmemo, setup_merge, detect_audio_paths, download_model_files
from metrics        import compute_metrics, print_metrics, print_summary
from visualization  import plot_cross_dataset_comparison
from spot_checks    import run_spot_checks

print(f'EVAL_DIR  : {EVAL_DIR}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'MODELS_DIR: {MODELS_DIR}')

## Section 2 — Model download

In [ ]:
MODEL_URLS = {
    # Backbones (shared with other Essentia notebooks)
    'msd-musicnn-1.pb': (
        'https://essentia.upf.edu/models/feature-extractors/musicnn/msd-musicnn-1.pb'
    ),
    'audioset-vggish-3.pb': (
        'https://essentia.upf.edu/models/feature-extractors/vggish/audioset-vggish-3.pb'
    ),
    # MuSE regression heads
    'muse-msd-musicnn-2.pb': (
        'https://essentia.upf.edu/models/classification-heads/muse/muse-msd-musicnn-2.pb'
    ),
    'muse-audioset-vggish-2.pb': (
        'https://essentia.upf.edu/models/classification-heads/muse/muse-audioset-vggish-2.pb'
    ),
}

download_model_files(MODEL_URLS, MODELS_DIR)
print('\nModels ready:')
for f in sorted(MODELS_DIR.glob('*.pb')):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

## Section 3 — Backbone + Head classes

In [ ]:
import essentia
import essentia.standard as es

essentia.log.infoActive = False
essentia.log.warningActive = False

# ── Constants ────────────────────────────────────────────────────────────────
_ALPHA = 0.13  # key-correction weight (validated in 01c/01d)

# ── Backbone configs ─────────────────────────────────────────────────────────
BACKBONES = {
    'musicnn': {
        'path':   MODELS_DIR / 'msd-musicnn-1.pb',
        'output': 'model/dense/BiasAdd',
    },
    'vggish': {
        'path':   MODELS_DIR / 'audioset-vggish-3.pb',
        'output': 'model/vggish/embeddings',
    },
}

# ── MuSE head configs ─────────────────────────────────────────────────────────
MUSE_MODELS = {
    'muse-musicnn': ('musicnn', MODELS_DIR / 'muse-msd-musicnn-2.pb'),
    'muse-vggish':  ('vggish',  MODELS_DIR / 'muse-audioset-vggish-2.pb'),
}


class EssentiaBackbone:
    """Extracts per-frame embeddings and key information from an audio file."""

    def __init__(self, backbone_name: str):
        cfg = BACKBONES[backbone_name]
        self._loader = es.MonoLoader(sampleRate=16000)
        if backbone_name == 'musicnn':
            self._extractor = es.TensorflowPredictMusiCNN(
                graphFilename=str(cfg['path']),
                output=cfg['output'],
            )
        else:  # vggish
            self._extractor = es.TensorflowPredictVGGish(
                graphFilename=str(cfg['path']),
                output=cfg['output'],
            )
        self._key_extractor = es.KeyExtractor()

    def extract(self, audio_path: Path):
        """Returns (embeddings, is_major: float, key_strength: float)."""
        audio       = self._loader(filename=str(audio_path))
        key, scale, key_strength = self._key_extractor(audio)
        embeddings  = self._extractor(audio)
        is_major    = 1.0 if scale == 'major' else 0.0
        return embeddings, is_major, float(key_strength)


class EssentiaHead:
    """Regression head: runs on pre-extracted embeddings → (valence, arousal) in [0, 1]."""

    def __init__(self, model_path: Path):
        self._model = es.TensorflowPredict2D(
            graphFilename=str(model_path),
            output='model/Identity',
        )

    def predict(self, embeddings) -> tuple:
        out     = self._model(embeddings)           # shape: (N_frames, 2)
        mean_v  = float(np.mean(out[:, 0]))
        mean_a  = float(np.mean(out[:, 1]))
        valence = float(np.clip((mean_v - 1.0) / 8.0, 0.0, 1.0))
        arousal = float(np.clip((mean_a - 1.0) / 8.0, 0.0, 1.0))
        return valence, arousal


def apply_key_correction(valence: float, is_major: float, key_strength: float) -> float:
    """Apply α-scaled key/mode correction to a raw valence prediction."""
    return float(np.clip(valence + _ALPHA * (is_major - 0.5) * key_strength, 0.0, 1.0))


print('Classes defined.')
print(f'  Key-correction α = {_ALPHA}')
print(f'  Backbones: {list(BACKBONES)}')
print(f'  MuSE heads: {list(MUSE_MODELS)}')

## Section 4 — Datasets

In [ ]:
deam_df,  deam_id,  deam_val,  deam_aro  = setup_deam(DATA_DIR)
pmemo_df, pmemo_id, pmemo_val, pmemo_aro = setup_pmemo(DATA_DIR)
merge_df, merge_id, merge_val, merge_aro = setup_merge(DATA_DIR)

audio_paths = detect_audio_paths(DATA_DIR)

DATASETS = {
    'DEAM':  (deam_df,  deam_id,  deam_val,  deam_aro,  audio_paths.get('deam')),
    'PMEmo': (pmemo_df, pmemo_id, pmemo_val, pmemo_aro, audio_paths.get('pmemo')),
    'MERGE': (merge_df, merge_id, merge_val, merge_aro, audio_paths.get('merge')),
}

for name, (df, *_) in DATASETS.items():
    status = f'{len(df)} tracks' if df is not None else 'unavailable'
    print(f'  {name}: {status}')

## Section 5 — Evaluation

### Phase 1 — Backbone extraction (embeddings + key info)

Each audio file is processed once per backbone. Results are cached in memory to avoid
re-loading audio during head evaluation.

In [ ]:
# backbone_cache[backbone_name][dataset_name] = list of
#   {'song_id': str, 'embeddings': np.ndarray,
#    'is_major': float, 'key_strength': float,
#    'gt_valence': float, 'gt_arousal': float}

backbone_cache = {}

for bb_name in BACKBONES:
    print(f'\n── Backbone: {bb_name} ──')
    backbone = EssentiaBackbone(bb_name)
    backbone_cache[bb_name] = {}

    for ds_name, (df, id_col, val_col, aro_col, audio_dir) in DATASETS.items():
        if df is None or audio_dir is None:
            print(f'  {ds_name}: skipped (unavailable)')
            continue

        audio_dir = Path(audio_dir)
        records = []
        t0 = time.time()

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f'{bb_name} / {ds_name}',
                           leave=False):
            song_id = str(row[id_col])
            # Locate audio file
            candidates = list(audio_dir.rglob(f'{song_id}.*'))
            if not candidates:
                continue
            audio_path = candidates[0]

            try:
                embeddings, is_major, key_strength = backbone.extract(audio_path)
            except Exception as e:
                continue

            gt_v = float(row[val_col])
            gt_a = float(row[aro_col])
            # Normalise DEAM annotations from [1, 9] to [0, 1]
            if ds_name == 'DEAM':
                gt_v = (gt_v - 1.0) / 8.0
                gt_a = (gt_a - 1.0) / 8.0

            records.append({
                'song_id':     song_id,
                'embeddings':  embeddings,
                'is_major':    is_major,
                'key_strength': key_strength,
                'gt_valence':  gt_v,
                'gt_arousal':  gt_a,
            })

        elapsed = time.time() - t0
        backbone_cache[bb_name][ds_name] = records
        print(f'  {ds_name}: {len(records)} tracks extracted in {elapsed:.0f}s '
              f'({elapsed / max(len(records), 1):.1f} s/track)')

print('\nPhase 1 complete.')

### Phase 2 — Head evaluation (base + key-corrected)

In [ ]:
# all_rows accumulates one row per song per model per dataset
all_rows = []

CHECKPOINT = EVAL_DIR / 'muse_eval_checkpoint.pkl'
completed  = set()
if CHECKPOINT.exists():
    with open(CHECKPOINT, 'rb') as f:
        all_rows, completed = pickle.load(f)
    print(f'Resumed from checkpoint: {len(completed)} (model, dataset) pairs done.')

for model_name, (bb_name, head_path) in MUSE_MODELS.items():
    print(f'\n── Head: {model_name} ──')
    head = EssentiaHead(head_path)

    for ds_name, records in backbone_cache[bb_name].items():
        for variant in (model_name, f'{model_name}+key'):
            key = (variant, ds_name)
            if key in completed:
                print(f'  {variant} / {ds_name}: already done, skipping.')
                continue

            t0 = time.time()
            for rec in tqdm(records, desc=f'{variant} / {ds_name}', leave=False):
                valence, arousal = head.predict(rec['embeddings'])
                if '+key' in variant:
                    valence = apply_key_correction(
                        valence, rec['is_major'], rec['key_strength']
                    )
                all_rows.append({
                    'model':      variant,
                    'dataset':    ds_name,
                    'song_id':    rec['song_id'],
                    'gt_valence': rec['gt_valence'],
                    'gt_arousal': rec['gt_arousal'],
                    'valence':    valence,
                    'arousal':    arousal,
                })

            elapsed = time.time() - t0
            n = len(records)
            print(f'  {variant} / {ds_name}: {n} tracks in {elapsed:.0f}s '
                  f'({elapsed / max(n, 1):.2f} s/track)')
            completed.add(key)
            with open(CHECKPOINT, 'wb') as f:
                pickle.dump((all_rows, completed), f)

results_df = pd.DataFrame(all_rows)
print(f'\nPhase 2 complete. Total rows: {len(results_df)}')
print(results_df.groupby(['model', 'dataset']).size().to_string())

## Section 6 — Cross-model comparison

In [ ]:
MODEL_ORDER   = ['muse-musicnn', 'muse-musicnn+key', 'muse-vggish', 'muse-vggish+key']
DATASET_ORDER = ['DEAM', 'PMEmo', 'MERGE']

print('=' * 80)
print('  MUSE MODELS — BENCHMARK RESULTS')
print('=' * 80)

for ds_name in DATASET_ORDER:
    ds_df = results_df[results_df['dataset'] == ds_name]
    if ds_df.empty:
        continue
    n = ds_df[ds_df['model'] == MODEL_ORDER[0]].shape[0]
    print(f'\n── {ds_name}  (n={n}) ──')
    print(f"{'Model':<22} {'Dim':<8} {'MAE':>7} {'R²':>7} {'r':>7} {'τ':>7}")
    print('-' * 58)
    for model in MODEL_ORDER:
        m_df = ds_df[ds_df['model'] == model]
        if m_df.empty:
            continue
        for dim in ('valence', 'arousal'):
            m = compute_metrics(m_df, dim=dim)
            label = model if dim == 'valence' else ''
            print(f"{label:<22} {dim:<8} {m['mae']:>7.4f} {m['r2']:>7.4f} "
                  f"{m['pearson_r']:>7.4f} {m['kendall_tau']:>7.4f}")

# Summary: average Pearson r across datasets
print(f'\n{"-"*58}')
print('  Average Pearson r across DEAM / PMEmo / MERGE')
print(f"{'Model':<22} {'V r (avg)':>10} {'A r (avg)':>10}")
print('-' * 44)
for model in MODEL_ORDER:
    m_df = results_df[results_df['model'] == model]
    v_rs, a_rs = [], []
    for ds in DATASET_ORDER:
        sub = m_df[m_df['dataset'] == ds]
        if sub.empty:
            continue
        v_rs.append(compute_metrics(sub, 'valence')['pearson_r'])
        a_rs.append(compute_metrics(sub, 'arousal')['pearson_r'])
    if v_rs:
        print(f"{model:<22} {np.mean(v_rs):>10.4f} {np.mean(a_rs):>10.4f}")

In [ ]:
# Scatter plots: predicted vs ground-truth for each variant × dataset
fig, axes = plt.subplots(
    len(MODEL_ORDER), len(DATASET_ORDER),
    figsize=(4 * len(DATASET_ORDER), 3.5 * len(MODEL_ORDER)),
)

for i, model in enumerate(MODEL_ORDER):
    for j, ds_name in enumerate(DATASET_ORDER):
        ax  = axes[i][j]
        sub = results_df[(results_df['model'] == model) &
                         (results_df['dataset'] == ds_name)]
        if sub.empty:
            ax.set_visible(False)
            continue
        ax.scatter(sub['gt_valence'], sub['valence'], alpha=0.2, s=3, color='steelblue')
        ax.plot([0, 1], [0, 1], 'r--', linewidth=0.8)
        m = compute_metrics(sub, 'valence')
        ax.set_title(f'{model}\n{ds_name}  r={m["pearson_r"]:.3f}', fontsize=8)
        ax.set_xlabel('GT valence', fontsize=7)
        ax.set_ylabel('Pred valence', fontsize=7)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

plt.suptitle('MuSE Models — Predicted vs GT Valence', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(EVAL_DIR / 'muse_valence_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Key-correction delta: how much does +key improve valence Pearson r?
print('Key-correction Δ valence Pearson r (base → +key):')
print(f"{'Backbone':<12} {'Dataset':<8} {'base r':>8} {'+key r':>8} {'Δ':>8}")
print('-' * 50)
for bb in ('muse-musicnn', 'muse-vggish'):
    for ds_name in DATASET_ORDER:
        base_df = results_df[(results_df['model'] == bb) &
                             (results_df['dataset'] == ds_name)]
        key_df  = results_df[(results_df['model'] == f'{bb}+key') &
                             (results_df['dataset'] == ds_name)]
        if base_df.empty or key_df.empty:
            continue
        r_base = compute_metrics(base_df, 'valence')['pearson_r']
        r_key  = compute_metrics(key_df,  'valence')['pearson_r']
        delta  = r_key - r_base
        sign   = '+' if delta > 0 else ''
        print(f"{bb:<12} {ds_name:<8} {r_base:>8.4f} {r_key:>8.4f} {sign}{delta:>7.4f}")

## Section 7 — Save results

In [ ]:
out_path = EVAL_DIR / 'muse_models_results.csv'
results_df.to_csv(out_path, index=False)
print(f'Saved {len(results_df)} rows → {out_path}')
print(results_df.groupby(['model', 'dataset']).size())

## Section 8 — Summary

In [ ]:
print('=' * 70)
print('  MUSE BENCHMARK — FULL SUMMARY')
print('=' * 70)

for model in MODEL_ORDER:
    m_df = results_df[results_df['model'] == model]
    if m_df.empty:
        continue
    print(f'\n  {model}')
    print(f"  {'Dataset':<8} {'V MAE':>7} {'V R²':>7} {'V r':>7} {'V τ':>7} "
          f"| {'A MAE':>7} {'A R²':>7} {'A r':>7} {'A τ':>7}")
    print('  ' + '-' * 68)
    for ds_name in DATASET_ORDER:
        sub = m_df[m_df['dataset'] == ds_name]
        if sub.empty:
            continue
        v = compute_metrics(sub, 'valence')
        a = compute_metrics(sub, 'arousal')
        print(f"  {ds_name:<8} {v['mae']:>7.4f} {v['r2']:>7.4f} {v['pearson_r']:>7.4f} "
              f"{v['kendall_tau']:>7.4f} | "
              f"{a['mae']:>7.4f} {a['r2']:>7.4f} {a['pearson_r']:>7.4f} "
              f"{a['kendall_tau']:>7.4f}")

print('\n' + '=' * 70)
print('  COMPARISON vs production model (deam-musicnn+key, from 01d)')
print('  Reference: DEAM  V r=0.7505 / A r=0.8650')
print('             PMEmo V r=0.6855 / A r=0.7240')
print('             MERGE V r=0.2863 / A r=0.6338')
print('=' * 70)